# ROSSA pipeline on a Hugging Face model (Google Colab)

Same ROSSA extraction as before, but the LLM is an **open Hugging Face model running on Colab's GPU** instead of Claude. Nothing runs on my laptop.

Paper: **Doyard 2016**. Model: **Qwen2.5-7B-Instruct** (open, no login, 4-bit so it fits the free T4 GPU).

**Before running:** Runtime -> Change runtime type -> **T4 GPU**. Then Runtime -> **Run all**. Takes about 10 minutes.

It pulls my prompt files from my GitHub repo, clears out the old Claude results so this is a clean open-model run, sends each ROSSA section only the part of the paper it needs, then runs my own `process` step to verify every quote and write the review.

In [ ]:
# 1. Install deps (torch is already in Colab). json-repair fixes small JSON slips from smaller models.
!pip -q install -U transformers accelerate bitsandbytes json-repair

In [ ]:
# 2. Get my pipeline + prompt files from GitHub, then clear the old Claude outputs so this run is clean
!rm -rf rossa-llm-pipeline
!git clone -q https://github.com/lUIZhMACEDO/rossa-llm-pipeline
%cd rossa-llm-pipeline

PAPER = "doyard_2016"
SECTIONS = ["study_information", "investigators", "animal_experimentation"]

# keep the prompt files; remove the committed Claude responses/review so they don't leak into this run
!rm -f output/{PAPER}.*.response.json output/{PAPER}.review.md output/{PAPER}.entry.json
print("prompt files kept:")
!ls output/*.prompt.json

In [ ]:
# 3. Load the open model (4-bit so it fits the free T4 GPU)
import time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 7B fits because each section only sees the part of the paper it needs (step 4).
# Faster but weaker option: Qwen/Qwen2.5-3B-Instruct
MODEL = "Qwen/Qwen2.5-7B-Instruct"

t0 = time.time()
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")
LOAD_S = round(time.time() - t0, 1)
print("loaded", MODEL, "in", LOAD_S, "s")

In [ ]:
# 4. Run the model on each section and save its JSON reply
import json, re
from pathlib import Path
from json_repair import repair_json

# Open models need the exact output shape shown to them, not just described.
FORMAT_RULES = '''

=== OUTPUT FORMAT (follow exactly) ===
Return ONE JSON object that fills in the output_template shown above.
EVERY leaf field MUST be an object with exactly these three keys: "value", "evidence", "confidence".
- "value": the answer, or null if the article does not state it.
- "evidence": an exact quote copied word for word from article_text that supports the value, or null.
- "confidence": one of "high", "medium", "low", "not_found".
Do NOT return a bare value. Do NOT rename, add, or drop top-level keys.
Keep the exact field names and nesting of output_template.
Reply with the JSON object ONLY - no prose, no markdown, no code fences.

Correctly formatted field (made-up example; take real values only from article_text):
  "doi": {"value": "10.0000/example.12345", "evidence": "doi:10.0000/example.12345", "confidence": "high"}
Field not found in the article:
  "pubmed_id": {"value": null, "evidence": null, "confidence": "not_found"}
'''

# Give each section only the part of the paper it needs. Less text = better answers and it lets
# the 7B model fit on the free GPU. (Checked on Doyard 2016: every quote Claude verified is inside.)
HEAD = re.compile(r'(?im)^\s*(abstract|introduction|background|materials? and methods|methods|'
                  r'experimental procedures|results|discussion|conclusions?)\s*$')

def window(text, section, cap=12000):
    heads = [(m.start(), m.group(1).lower()) for m in HEAD.finditer(text)]
    def pos(*names):
        for p, n in heads:
            if n.startswith(names):
                return p
        return None
    intro = pos('introduction', 'background')
    front = text[:intro] if intro else text[:6000]
    if section in ('study_information', 'investigators'):
        return front[:cap]          # title, authors, email, abstract, funding, conflicts
    meth = pos('materials', 'methods', 'experimental')
    if meth is None:
        return text[:cap]
    later = [p for p, _ in heads if p > meth]
    meth_end = later[0] if later else len(text)
    start = pos('abstract') or 0
    head = text[start:intro] if intro and intro > start else text[start:start + 3000]
    methods = text[meth:meth_end][:cap - len(head)]
    # abstract + methods always; the introduction only if there is room (some journals put methods last)
    room = cap - len(head) - len(methods)
    if intro and intro < meth and room > 0:
        intro_txt = text[intro:min(meth, [p for p, _ in heads if p > intro][0])][:room]
        return head + intro_txt + methods
    return head + methods

CONF = {'high', 'medium', 'low', 'not_found'}

def is_leaf(t):
    return isinstance(t, dict) and set(t) == {'value', 'evidence', 'confidence'}

def coerce(tmpl, got):
    # reshape whatever the model returned into the exact output_template structure
    if is_leaf(tmpl):
        if isinstance(got, dict) and ('value' in got or 'evidence' in got or 'quote' in got):
            val, ev, conf = got.get('value'), got.get('evidence', got.get('quote')), got.get('confidence')
        else:
            val, ev, conf = got, None, None   # bare value
        if isinstance(val, str) and val.strip().lower() in ('', 'null', 'none', 'n/a', 'not found', 'not_found'):
            val = None
        if not isinstance(ev, (str, list)) or ev == '':
            ev = None
        if conf not in CONF:
            conf = 'not_found' if val is None else ('low' if ev is None else 'medium')
        return {'value': val, 'evidence': ev, 'confidence': conf}
    if isinstance(tmpl, dict):
        src = got if isinstance(got, dict) else {}
        out = {k: coerce(v, src.get(k)) for k, v in tmpl.items() if k != 'reviewer_flags'}
        if 'reviewer_flags' in tmpl:
            flags = src.get('reviewer_flags', [])
            out['reviewer_flags'] = [str(f) for f in flags] if isinstance(flags, list) else [str(flags)]
        return out
    if isinstance(tmpl, list):
        item = tmpl[0] if tmpl else None
        if isinstance(got, dict):
            got = [got]
        if not isinstance(got, list):
            return []
        return [coerce(item, g) for g in got if g] if item is not None else got
    return got

def unwrap(tmpl, obj, section):
    # peel off wrappers like {"output": ...} or {"study_information": ...}
    keys = set(tmpl) - {'reviewer_flags'}
    for _ in range(3):
        if isinstance(obj, dict) and keys & set(obj):
            return obj
        if isinstance(obj, list) and len(keys) == 1:
            return {next(iter(keys)): obj}
        if not isinstance(obj, dict):
            return obj
        inner = obj.get(section) or obj.get('output') or obj.get('result') or obj.get('data')
        if inner is None and len(obj) == 1:
            inner = next(iter(obj.values()))
        if inner is None:
            return obj
        flags = obj.get('reviewer_flags')
        obj = inner
        if flags and isinstance(obj, dict) and 'reviewer_flags' not in obj:
            obj['reviewer_flags'] = flags
    return obj

def run_section(section):
    prompt = json.loads(Path(f'output/{PAPER}.{section}.prompt.json').read_text())
    template = prompt['request']['output_template']
    request = dict(prompt['request'])
    request['article_text'] = window(request['article_text'], section)
    user_content = json.dumps(request, ensure_ascii=False) + FORMAT_RULES
    messages = [
        {'role': 'system', 'content': prompt['system_prompt']},
        {'role': 'user', 'content': user_content},
    ]
    t0 = time.time()
    enc = tok.apply_chat_template(messages, add_generation_prompt=True,
                                  return_tensors='pt', return_dict=True).to(model.device)
    out = model.generate(**enc, max_new_tokens=3072, do_sample=False)
    reply = tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
    TIMES[section] = {'seconds': round(time.time() - t0, 1),
                      'input_tokens': enc['input_ids'].shape[1],
                      'output_tokens': out.shape[1] - enc['input_ids'].shape[1]}
    Path(f'output/{PAPER}.{section}.raw.txt').write_text(reply)   # keep the raw reply for inspection
    obj = coerce(template, unwrap(template, json.loads(repair_json(reply)), section))
    obj['_meta'] = {'backend': 'huggingface', 'model': MODEL}
    Path(f'output/{PAPER}.{section}.response.json').write_text(json.dumps(obj, indent=2, ensure_ascii=False))
    filled = sum(1 for _ in json.dumps(obj).split('"value": ')[1:]) - json.dumps(obj).count('"value": null')
    print(f'[ok] {section}: {len(request["article_text"])} chars of paper in, {filled} fields filled')

TIMES = {'model_load_s': LOAD_S, 'gpu': torch.cuda.get_device_name(0)}
for s in SECTIONS:
    print('running', s, '...')
    run_section(s)
    print('   ', TIMES[s])
TIMES['total_s'] = round(sum(TIMES[s]['seconds'] for s in SECTIONS), 1)
Path(f'output/{PAPER}.huggingface.timing.json').write_text(json.dumps(TIMES, indent=2))
print('total model time:', TIMES['total_s'], 's')

In [ ]:
# 5. Run my own pipeline's process step: verify every quote against the paper, write the review
!python rossa_pipeline.py process {PAPER}

# keep a clearly-named copy of this (open-model) run
!cp output/{PAPER}.review.md output/{PAPER}.huggingface.review.md
!cp output/{PAPER}.entry.json output/{PAPER}.huggingface.entry.json

In [ ]:
# 6. Show the result (this open-model run)
from IPython.display import Markdown, display
display(Markdown(Path(f"output/{PAPER}.huggingface.review.md").read_text()))

In [ ]:
# 7. Download the results
from google.colab import files
files.download(f"output/{PAPER}.huggingface.review.md")
files.download(f"output/{PAPER}.huggingface.entry.json")
files.download(f"output/{PAPER}.huggingface.timing.json")